In [5]:
"""
Stage 5 (ES vs LS) — Final evaluation across SMOTE variants
=============================================================

Evaluates the two SMOTE variants from Stage 4 on the held-out test set
and produces a side-by-side comparison.

For each SMOTE variant in {balanced, imbalanced}:
  1. Read consensus_hyperparams.csv from the matching Stage-4 dir.
  2. Read train_postfeature_smote.csv (full-train SMOTE applied) +
     test_postfeature.csv (held-out, untouched).
  3. Re-instantiate sklearn pipelines (no SMOTE — already applied to the
     saved CSV) using consensus HPs and fit on the SMOTE-augmented
     full-train data.
  4. SAMPLE WEIGHTING (matches Stage 4):
       - Real samples       -> composite (sex x cell_type x class) weight
       - Synthetic samples  -> uniform 1.0
  5. Predict on the unmodified test set; compute the FULL metric battery
     on BOTH train and test, plus the train - test gap. The battery
     includes OVERALL precision plus PER-CLASS precision / recall / F1
     alongside the standard set.
  6. Tailored ENet / RFE gene subsets at >= INCLUSION_THR fold-inclusion
     if those CSVs exist; otherwise full panel.
  7. Q1-quality plots: ROC, PR, confusion matrices, metric heatmap,
     train-vs-test grouped bars + gap panel, per-class metrics bars.

Class encoding: NEG=LS=0 (minority), POS=ES=1 (majority).

Iterates over PANEL_SIZES = [5, 7, 10, 12, 15]. For each panel size,
both SMOTE variants are evaluated and a per-panel cross-variant
comparison folder is written. After all panels are done, a master
comparison aggregates every (panel_size, variant, classifier) row.

Outputs (per panel K, per variant V):
  Outputs/stage5_EVAL_ESLS_top{K}_smote_{V}/
  Outputs/stage5_EVAL_ESLS_top{K}_smote_comparison/
  Outputs/stage5_EVAL_ESLS_smote_master_comparison/
"""

from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
ESLS_DIR     = PIPELINE_DIR / "ES vs LS"
SPLIT_DIR    = ESLS_DIR / "Outputs" / "stage2_split_stratum_aware_ESLS"
OUTPUT_BASE  = ESLS_DIR / "Outputs"

META_PATH = ESLS_DIR / "Meta_data_ESLS.csv"

CONDITION_COL = "condition"
INCLUSION_THR = 80.0
NEG_LABEL     = "LS"
POS_LABEL     = "ES"
SEED          = 42

SMOTE_VARIANTS = ["balanced", "imbalanced"]
PANEL_SIZES    = [5, 7, 10, 12, 15]

INT_PARAMS = {"clf__n_neighbors", "clf__n_features_to_select"}

sns.set_style("whitegrid", {"grid.linestyle": ":", "grid.alpha": 0.5})
plt.rcParams.update({
    "savefig.dpi":       300, "figure.dpi": 100,
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         10, "axes.titlesize": 11,
    "axes.titleweight":  "bold", "axes.labelsize": 10,
    "axes.spines.top":   False, "axes.spines.right": False,
    "xtick.labelsize":   9, "ytick.labelsize": 9,
    "legend.fontsize":   8.5, "legend.frameon": False,
})

CLF_COLORS = {
    "mSVM-RBF":    "#0173B2",
    "mSVM-Linear": "#DE8F05",
    "mSVM-RFE":    "#029E73",
    "kNN":         "#CC78BC",
    "GaussianNB":  "#CA9161",
    "ElasticNet":  "#D55E00",
}
VARIANT_COLORS = {"balanced": "#1f77b4", "imbalanced": "#d62728"}
CLASS_NAMES = [NEG_LABEL, POS_LABEL]


# ============================ Data helpers =================================
def load_meta(sample_ids):
    meta = pd.read_csv(META_PATH)
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    return meta.loc[sample_ids]


def to_binary_labels(y_str):
    return np.where(np.asarray(y_str) == NEG_LABEL, NEG_LABEL, POS_LABEL)


def encode_binary(y_str_binary):
    return np.where(y_str_binary == POS_LABEL, 1, 0).astype(int)


def load_pf(path: Path):
    """Load post-feature CSV (genes x samples). ES vs LS gene IDs start with
    'ENSG' (full string e.g. 'ENSG00000000419.12|DPM1|protein_coding')."""
    df = pd.read_csv(path, index_col=0)
    looks_like_genes = df.index.astype(str).str.startswith("ENSG").any()
    if not looks_like_genes:
        df = df.T
    return df


def load_smote_train(s4_dir: Path):
    """Load the full-train SMOTE-augmented post-feature CSV + its metadata.
    Synthetic samples have no metadata in the original split; we recover
    only condition and synthetic-flag from the saved meta CSV."""
    train_df = load_pf(s4_dir / "train_postfeature_smote.csv")
    meta = pd.read_csv(s4_dir / "train_postfeature_smote_meta.csv")
    meta = meta.set_index("sample_id").loc[train_df.columns]
    return train_df, meta


# ============================ Sample-weight helpers =========================
def composite_sample_weights(meta: pd.DataFrame, y_int: np.ndarray) -> np.ndarray:
    strata = meta["sex"].astype(str) + "_" + meta["cell_type"].astype(str)
    s_counts = strata.value_counts()
    n, k = len(strata), len(s_counts)
    w_conf = strata.map(lambda s: (n / k) / s_counts[s]).values.astype(float)

    classes, c_counts = np.unique(y_int, return_counts=True)
    n_cls = len(classes)
    class_w = {c: n / (n_cls * cnt) for c, cnt in zip(classes, c_counts)}
    w_class = np.array([class_w[y] for y in y_int], dtype=float)

    w = w_conf * w_class
    return w / w.mean()


def extended_smote_weights(smote_meta_df: pd.DataFrame,
                             meta_train_real: pd.DataFrame,
                             y_train_int: np.ndarray) -> np.ndarray:
    """Per-sample weight vector for the SMOTE-augmented train block:
        * Real samples (synthetic == False) -> composite weight.
        * Synthetic samples (synthetic == True) -> uniform 1.0.
    Result is normalised to mean 1."""
    n = len(smote_meta_df)
    weights = np.ones(n, dtype=float)

    real_mask = ~smote_meta_df["synthetic"].astype(bool).values
    if real_mask.sum() == 0:
        return weights

    real_sample_ids = smote_meta_df.index[real_mask]
    real_meta       = meta_train_real.loc[real_sample_ids,
                                            ["sex", "cell_type"]].copy()
    real_y_int      = y_train_int[real_mask]
    real_w          = composite_sample_weights(real_meta, real_y_int)
    weights[real_mask] = real_w

    return weights / weights.mean()


# ============================ HP parsing ====================================
def _typed(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s in ("None", "nan", "NaN"):
        return None
    if s in ("True", "False"):
        return s == "True"
    if s.lower() in ("scale", "auto"):
        return s.lower()
    if s in ("uniform", "distance"):
        return s
    try:
        i = int(s)
        if str(i) == s:
            return i
    except ValueError:
        pass
    try:
        return float(s)
    except ValueError:
        return s


def consensus_hp_dict(consensus_df):
    out = {}
    cdf = consensus_df.set_index("classifier")
    for clf in cdf.index:
        params = {}
        for col in cdf.columns:
            if col.endswith("__support_pct"):
                continue
            v = _typed(cdf.loc[clf, col])
            if v is None:
                continue
            if col.startswith("smote__"):
                continue
            params[col] = v
        out[clf] = params
    return out


def load_inclusion_genes(path, threshold):
    if not path.exists():
        return []
    df = pd.read_csv(path)
    if "fold_inclusion_pct" not in df.columns or "gene" not in df.columns:
        return []
    return df.loc[df["fold_inclusion_pct"] >= threshold, "gene"].tolist()


# ============================ Classifier factory ============================
def build_pipelines(seed=SEED):
    """Sklearn pipelines (no SMOTE — synthetic samples already in the
    augmented training CSV). The RFE step is named `clf` for consistency
    with Stage 4's SMOTEWeightedPipeline wrapper, so the consensus HP keys
    apply directly without rewriting."""
    return {
        "mSVM-RBF": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="rbf", probability=True,
                                          random_state=seed))]),
            "supports_sw": True,
        },
        "mSVM-Linear": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="linear", probability=True,
                                          random_state=seed))]),
            "supports_sw": True,
        },
        "mSVM-RFE": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", RFE(SVC(kernel="linear",
                                              probability=True,
                                              random_state=seed)))]),
            "supports_sw": True,
        },
        "kNN": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", KNeighborsClassifier())]),
            "supports_sw": False,
        },
        "GaussianNB": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", GaussianNB())]),
            "supports_sw": True,
        },
        "ElasticNet": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", LogisticRegression(
                                  penalty="elasticnet", solver="saga",
                                  max_iter=10000, tol=1e-4,
                                  random_state=seed))]),
            "supports_sw": True,
        },
    }


# ============================ Metrics =======================================
def evaluate_binary(y_true, y_pred, y_proba_pos):
    """Full metric battery — overall + per-class. See stage5_EVAL_ESLS.py."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    prec_per_class = precision_score(y_true, y_pred, average=None,
                                     labels=[0, 1], zero_division=0)
    rec_per_class  = recall_score(y_true, y_pred, average=None,
                                  labels=[0, 1], zero_division=0)
    f1_per_class   = f1_score(y_true, y_pred, average=None,
                              labels=[0, 1], zero_division=0)

    return {
        "balanced_accuracy":  balanced_accuracy_score(y_true, y_pred),
        "macro_precision":    precision_score(y_true, y_pred,
                                              average="macro", zero_division=0),
        "macro_recall":       recall_score(y_true, y_pred,
                                           average="macro", zero_division=0),
        "macro_f1":           f1_score(y_true, y_pred, average="macro",
                                       zero_division=0),
        "auroc":              roc_auc_score(y_true, y_proba_pos),
        "pr_auc":             average_precision_score(y_true, y_proba_pos),
        "sensitivity":        sens,
        "specificity":        spec,
        "mcc":                matthews_corrcoef(y_true, y_pred),
        "brier":              brier_score_loss(y_true, y_proba_pos),
        "cohen_kappa":        cohen_kappa_score(y_true, y_pred),
        f"precision_{NEG_LABEL}": float(prec_per_class[0]),
        f"precision_{POS_LABEL}": float(prec_per_class[1]),
        f"recall_{NEG_LABEL}":    float(rec_per_class[0]),
        f"recall_{POS_LABEL}":    float(rec_per_class[1]),
        f"f1_{NEG_LABEL}":        float(f1_per_class[0]),
        f"f1_{POS_LABEL}":        float(f1_per_class[1]),
    }


# ============================ Plotting ======================================
def plot_roc(predictions, out_path, title_suffix):
    fig, ax = plt.subplots(figsize=(8, 7))
    for clf_name, p in predictions.items():
        y_true = np.asarray(p["y_true"])
        y_pos  = np.asarray(p["y_proba_pos"])
        fpr, tpr, _ = roc_curve(y_true, y_pos)
        auc = roc_auc_score(y_true, y_pos)
        ax.plot(fpr, tpr, color=CLF_COLORS[clf_name], linewidth=1.8,
                label=f"{clf_name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("1 − Specificity (FPR)"); ax.set_ylabel("Sensitivity (TPR)")
    ax.set_title(f"ROC on held-out test — {title_suffix}", fontweight="bold")
    ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_pr(predictions, out_path, title_suffix):
    fig, ax = plt.subplots(figsize=(8, 7))
    for clf_name, p in predictions.items():
        y_true = np.asarray(p["y_true"])
        y_pos  = np.asarray(p["y_proba_pos"])
        prec, rec, _ = precision_recall_curve(y_true, y_pos)
        ap = average_precision_score(y_true, y_pos)
        ax.plot(rec, prec, color=CLF_COLORS[clf_name], linewidth=1.8,
                label=f"{clf_name} (AP={ap:.3f})")
    prevalence = np.mean([np.mean(np.asarray(p["y_true"]))
                          for p in predictions.values()])
    ax.axhline(prevalence, color="gray", linestyle="--", linewidth=1,
                alpha=0.7, label=f"prevalence = {prevalence:.2f}")
    ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR on held-out test — {title_suffix}", fontweight="bold")
    ax.legend(loc="lower left")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_confusion_grid(predictions, out_path, title_suffix):
    n = len(predictions); cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.2 * rows),
                              constrained_layout=True)
    axes = np.array(axes).ravel()
    for ax, (clf_name, p) in zip(axes, predictions.items()):
        y_true = np.asarray(p["y_true"]); y_pred = np.asarray(p["y_pred"])
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
        sns.heatmap(cm_norm, annot=cm.astype(int), fmt="d",
                     cmap="Blues", vmin=0, vmax=1, cbar=False, square=True,
                     xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                     annot_kws={"fontsize": 11, "fontweight": "bold"}, ax=ax)
        for i in range(2):
            for j in range(2):
                ax.text(j + 0.5, i + 0.78, f"{cm_norm[i, j]*100:.0f}%",
                        ha="center", va="center", fontsize=8.5,
                        color="white" if cm_norm[i, j] > 0.5 else "#444")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(clf_name)
    for ax in axes[len(predictions):]:
        ax.axis("off")
    fig.suptitle(f"Confusion matrices — {title_suffix}", fontweight="bold")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_metric_heatmap(metrics_df, out_path, title_suffix):
    cols = ["balanced_accuracy", "macro_precision", "macro_recall", "macro_f1",
            "auroc", "pr_auc", "sensitivity", "specificity", "mcc",
            "cohen_kappa", "brier",
            f"precision_{NEG_LABEL}", f"precision_{POS_LABEL}",
            f"recall_{NEG_LABEL}",    f"recall_{POS_LABEL}",
            f"f1_{NEG_LABEL}",        f"f1_{POS_LABEL}"]
    pretty = ["Bal. acc.", "macro-Prec.", "macro-Rec.", "macro-F1",
              "AUROC", "PR-AUC", "Sens. (ES)", "Spec. (LS)", "MCC",
              "Cohen's κ", "Brier (↓)",
              f"Prec. {NEG_LABEL}", f"Prec. {POS_LABEL}",
              f"Rec. {NEG_LABEL}",  f"Rec. {POS_LABEL}",
              f"F1 {NEG_LABEL}",    f"F1 {POS_LABEL}"]
    M = metrics_df.set_index("classifier")[cols].copy()
    M.columns = pretty
    norm_M = M.copy()
    norm_M["Brier (↓)"] = 1 - norm_M["Brier (↓)"].clip(0, 1)
    fig, ax = plt.subplots(figsize=(15, 4.8))
    sns.heatmap(norm_M, annot=M.round(3), fmt="", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, linecolor="white",
                cbar_kws={"shrink": 0.7}, annot_kws={"fontsize": 8.5}, ax=ax)
    ax.set_title(f"Metric battery — {title_suffix}", fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_per_class_metrics(metrics_df: pd.DataFrame, out_path: Path,
                           title_suffix: str):
    rows = []
    for _, r in metrics_df.iterrows():
        for cls in (NEG_LABEL, POS_LABEL):
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "Precision",
                         "score": float(r[f"precision_{cls}"])})
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "Recall",
                         "score": float(r[f"recall_{cls}"])})
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "F1",
                         "score": float(r[f"f1_{cls}"])})
    long_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                             constrained_layout=True, sharey=True)
    for ax, m in zip(axes, ["Precision", "Recall", "F1"]):
        sub = long_df[long_df["metric"] == m]
        sns.barplot(data=sub, x="classifier", y="score", hue="class",
                    palette={NEG_LABEL: "#d62728", POS_LABEL: "#1f77b4"},
                    ax=ax, edgecolor="white")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", fontsize=7.5, padding=1)
        ax.set_ylim([0, 1.05]); ax.set_ylabel("Score")
        ax.set_xlabel(""); ax.set_title(m, fontweight="bold")
        plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
        if ax.get_legend(): ax.get_legend().set_title("Class")
    fig.suptitle(f"Per-class metrics on held-out test — {title_suffix}",
                 fontweight="bold")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_train_vs_test(train_df, test_df, out_path, title_suffix):
    headline = ["balanced_accuracy", "macro_precision", "macro_recall",
                "macro_f1", "auroc", "pr_auc", "mcc", "cohen_kappa"]
    pretty = {"balanced_accuracy": "Bal. acc.",
              "macro_precision":   "macro-Prec.",
              "macro_recall":      "macro-Rec.",
              "macro_f1":          "macro-F1",
              "auroc":             "AUROC",
              "pr_auc":            "PR-AUC",
              "mcc":               "MCC",
              "cohen_kappa":       "Cohen's κ"}
    tr = train_df.set_index("classifier")[headline].copy()
    te =  test_df.set_index("classifier")[headline].copy()
    gap = tr - te
    metric_order = [pretty[h] for h in headline]
    clf_order = list(CLF_COLORS)

    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0], hspace=0.35)
    ax_top = fig.add_subplot(gs[0]); ax_bot = fig.add_subplot(gs[1])

    width = 0.06; x = np.arange(len(metric_order))
    for ci, clf in enumerate(clf_order):
        offset = (ci - len(clf_order) / 2 + 0.5) * width * 2
        tr_vals = [float(tr.loc[clf, h]) for h in headline]
        te_vals = [float(te.loc[clf, h]) for h in headline]
        ax_top.bar(x + offset - width / 2, tr_vals, width=width,
                   color=CLF_COLORS[clf], edgecolor="white", linewidth=0.5,
                   alpha=0.95)
        ax_top.bar(x + offset + width / 2, te_vals, width=width,
                   color=CLF_COLORS[clf], edgecolor="white", linewidth=0.5,
                   hatch="///", alpha=0.95)
    ax_top.set_xticks(x); ax_top.set_xticklabels(metric_order)
    ax_top.set_ylim([min(0, min(tr.values.min(), te.values.min()) - 0.1), 1.05])
    ax_top.axhline(0, color="black", linewidth=0.6)
    ax_top.set_ylabel("Score")
    ax_top.set_title(f"Train vs Test — {title_suffix}", fontweight="bold")
    handles = [Patch(facecolor=CLF_COLORS[c], edgecolor="white", label=c)
               for c in clf_order]
    handles += [Patch(facecolor="white", edgecolor="black", label="Train"),
                Patch(facecolor="white", edgecolor="black",
                       hatch="///", label="Test")]
    ax_top.legend(handles=handles, loc="upper left",
                   bbox_to_anchor=(1.01, 1.0))

    gap_long = []
    for clf in tr.index:
        for metric in headline:
            gap_long.append({"classifier": clf, "metric": pretty[metric],
                              "gap": float(gap.loc[clf, metric])})
    gap_long_df = pd.DataFrame(gap_long)
    sns.barplot(data=gap_long_df, x="metric", y="gap",
                 hue="classifier", hue_order=clf_order,
                 order=metric_order, palette=CLF_COLORS,
                 edgecolor="white", linewidth=0.6, ax=ax_bot)
    for container in ax_bot.containers:
        ax_bot.bar_label(container, fmt="%+.2f", padding=2,
                          fontsize=7, label_type="edge")
    ax_bot.axhline(0, color="black", linewidth=0.8)
    ax_bot.set_ylabel("Train − Test gap"); ax_bot.set_xlabel("")
    ax_bot.set_title("Generalisation gap (positive = train > test = overfit)",
                      fontweight="bold")
    if ax_bot.get_legend():
        ax_bot.get_legend().remove()
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_variant_comparison(metrics_by_variant, out_path, title_suffix=""):
    """Grouped bar chart per classifier; hued by SMOTE variant. Headline
    metrics now include macro-precision."""
    headline = ["balanced_accuracy", "macro_precision", "macro_f1",
                "auroc", "pr_auc", "mcc", "cohen_kappa"]
    pretty   = {"balanced_accuracy": "Bal. acc.",
                "macro_precision":   "macro-Prec.",
                "macro_f1":          "macro-F1",
                "auroc":             "AUROC",
                "pr_auc":            "PR-AUC",
                "mcc":               "MCC",
                "cohen_kappa":       "κ"}

    rows = []
    for variant, df in metrics_by_variant.items():
        for _, r in df.iterrows():
            for h in headline:
                rows.append({"variant": variant,
                              "classifier": r["classifier"],
                              "metric": pretty[h],
                              "score": float(r[h])})
    long_df = pd.DataFrame(rows)

    n_clf = len(CLF_COLORS); cols = 3
    rows_n = int(np.ceil(n_clf / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3.6 * rows_n),
                              constrained_layout=True)
    axes = np.array(axes).ravel()
    for ax, clf in zip(axes, CLF_COLORS.keys()):
        sub = long_df[long_df["classifier"] == clf]
        sns.barplot(data=sub, x="metric", y="score", hue="variant",
                     palette=VARIANT_COLORS, ax=ax, edgecolor="white")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", fontsize=7, padding=1)
        ax.set_ylim([0, 1.05]); ax.set_ylabel("Score"); ax.set_xlabel("")
        ax.set_title(clf)
        if ax.get_legend(): ax.get_legend().remove()
    for ax in axes[len(CLF_COLORS):]:
        ax.axis("off")
    fig.legend(handles=[Patch(facecolor=VARIANT_COLORS[v], label=v.upper())
                         for v in SMOTE_VARIANTS],
                loc="upper right", title="SMOTE variant")
    suffix = f" — {title_suffix}" if title_suffix else ""
    fig.suptitle(f"SMOTE variant comparison on held-out test (ES vs LS){suffix}",
                  fontweight="bold")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


# ============================ Per-variant run ===============================
def run_variant(variant: str, panel_size: int):
    s4_dir   = OUTPUT_BASE / f"stage4_HP_ESLS_top{panel_size}_smote_{variant}"
    out_dir  = OUTPUT_BASE / f"stage5_EVAL_ESLS_top{panel_size}_smote_{variant}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"VARIANT: SMOTE = {variant.upper()}  (top {panel_size})")
    print(f"Stage-4 source: {s4_dir.name}")
    print(f"Output dir    : {out_dir.name}")
    print("=" * 72)

    if not s4_dir.exists():
        raise FileNotFoundError(
            f"{s4_dir} not found. Run stage4_HP_ESLS_smote.py first.")

    # ---- Load full-train SMOTE-augmented CSV + held-out test ----
    train_df, train_meta = load_smote_train(s4_dir)
    test_df  = load_pf(s4_dir / "test_postfeature.csv")
    print(f"Train (SMOTE-augmented): {train_df.shape[1]} samples x "
           f"{train_df.shape[0]} genes "
           f"({(train_meta['synthetic']==True).sum()} synthetic)")
    print(f"Test (held-out)        : {test_df.shape[1]} samples x "
           f"{test_df.shape[0]} genes")

    full_genes = train_df.index.tolist()

    y_train = encode_binary(to_binary_labels(train_meta[CONDITION_COL].values))

    meta_test = load_meta(test_df.columns)
    y_test    = encode_binary(to_binary_labels(meta_test[CONDITION_COL].values))

    print(f"Train class balance: LS={int((y_train==0).sum())}, "
           f"ES={int((y_train==1).sum())}")
    print(f"Test  class balance: LS={int((y_test==0).sum())}, "
           f"ES={int((y_test==1).sum())}")

    X_train_full = train_df.T.values.astype(np.float64)
    X_test_full  = test_df.T.values.astype(np.float64)

    # ---- Composite confounder weights for REAL samples + uniform 1.0 for
    # SYNTHETIC samples (normalised to mean 1).
    # Real-sample metadata comes from Meta_data_ESLS.csv; synthetic samples
    # have only condition + synthetic flag in the saved meta CSV.
    real_sample_ids = train_meta.index[~train_meta["synthetic"].astype(bool)]
    meta_train_real = load_meta(real_sample_ids)
    ext_sample_weights = extended_smote_weights(
        train_meta, meta_train_real, y_train)
    n_real  = int((~train_meta["synthetic"].astype(bool)).sum())
    n_synth = int(train_meta["synthetic"].astype(bool).sum())
    print(f"Sample weights: real={n_real} (composite sex x cell_type x class), "
          f"synthetic={n_synth} (uniform 1.0)  -> mean={ext_sample_weights.mean():.3f}, "
          f"min={ext_sample_weights.min():.3f}, max={ext_sample_weights.max():.3f}")

    consensus = pd.read_csv(s4_dir / "consensus_hyperparams.csv")
    hp_dict   = consensus_hp_dict(consensus)

    enet_inc = s4_dir / "enet_fold_inclusion.csv"
    rfe_inc  = s4_dir / "rfe_fold_inclusion.csv"
    enet_genes = [g for g in load_inclusion_genes(enet_inc, INCLUSION_THR)
                   if g in full_genes]
    rfe_genes  = [g for g in load_inclusion_genes(rfe_inc, INCLUSION_THR)
                   if g in full_genes]
    tailored = {"ElasticNet": enet_genes, "mSVM-RFE": rfe_genes}
    print(f"\nTailored gene subsets at fold_inclusion_pct >= {INCLUSION_THR}%:")
    for k, gs in tailored.items():
        print(f"  {k}: {len(gs)} genes")

    pipelines = build_pipelines(SEED)
    predictions, metrics_rows, train_metrics_rows, gap_rows = {}, [], [], []
    hp_rows, gene_subset_rows = [], []

    for clf_name, cfg in pipelines.items():
        if clf_name in tailored and len(tailored[clf_name]) >= 2:
            genes_used = tailored[clf_name]
            subset_label = f"tailored ({len(genes_used)} >= {INCLUSION_THR}%)"
        else:
            if clf_name in tailored:
                print(f"  [{clf_name}] WARN: tailored set empty/<2; using full panel")
            genes_used = full_genes
            subset_label = f"full panel ({len(genes_used)} genes)"

        gene_idx = [full_genes.index(g) for g in genes_used]
        X_tr = X_train_full[:, gene_idx]
        X_te = X_test_full[:,  gene_idx]
        gene_subset_rows.append({"classifier":  clf_name,
                                  "subset_kind": subset_label,
                                  "n_genes":     len(genes_used),
                                  "genes":       ";".join(genes_used)})

        params = hp_dict.get(clf_name, {})
        valid_keys = set(cfg["pipe"].get_params().keys())
        applied = {k: v for k, v in params.items() if k in valid_keys}

        for ip in INT_PARAMS & set(applied.keys()):
            if applied[ip] is not None:
                applied[ip] = int(float(applied[ip]))
        if clf_name == "mSVM-RFE" and "clf__n_features_to_select" in applied:
            requested = int(applied["clf__n_features_to_select"])
            applied["clf__n_features_to_select"] = max(1,
                min(requested, len(genes_used)))

        cfg["pipe"].set_params(**applied)
        supports_sw = cfg.get("supports_sw", False)
        hp_rows.append({"classifier": clf_name,
                         "applied_params": str(applied),
                         "subset_kind": subset_label,
                         "n_genes": len(genes_used),
                         "sample_weight_used": supports_sw})

        if supports_sw:
            cfg["pipe"].fit(X_tr, y_train,
                             clf__sample_weight=ext_sample_weights)
        else:
            cfg["pipe"].fit(X_tr, y_train)

        # Train metrics (informational; SMOTE-augmented)
        y_train_pred  = cfg["pipe"].predict(X_tr)
        y_train_proba = cfg["pipe"].predict_proba(X_tr)[:, 1]
        m_train = evaluate_binary(y_train, y_train_pred, y_train_proba)
        m_train["classifier"] = clf_name
        train_metrics_rows.append(m_train)

        # Test metrics (real held-out)
        y_pred  = cfg["pipe"].predict(X_te)
        y_proba_pos = cfg["pipe"].predict_proba(X_te)[:, 1]
        m = evaluate_binary(y_test, y_pred, y_proba_pos)
        m["classifier"] = clf_name
        metrics_rows.append(m)

        gap_row = {"classifier": clf_name}
        for k, v_train in m_train.items():
            if k == "classifier": continue
            v_test = m.get(k)
            if isinstance(v_train, (int, float)) and isinstance(v_test, (int, float)):
                gap_row[f"{k}_train"] = v_train
                gap_row[f"{k}_test"]  = v_test
                gap_row[f"{k}_gap"]   = v_train - v_test
        gap_rows.append(gap_row)

        predictions[clf_name] = {"y_true": y_test.tolist(),
                                  "y_pred": y_pred.tolist(),
                                  "y_proba_pos": y_proba_pos.tolist()}

        print(f"\n[{clf_name}] {subset_label} | applied: {applied}")
        print(f"  TRAIN(+SMOTE) BA={m_train['balanced_accuracy']:.3f} "
               f"AUROC={m_train['auroc']:.3f}  "
               f"macro-Prec={m_train['macro_precision']:.3f} "
               f"F1={m_train['macro_f1']:.3f}")
        print(f"  TEST          BA={m['balanced_accuracy']:.3f} "
               f"AUROC={m['auroc']:.3f}  "
               f"macro-Prec={m['macro_precision']:.3f} "
               f"F1={m['macro_f1']:.3f} MCC={m['mcc']:.3f}")
        print(f"  TEST per-class — "
               f"{NEG_LABEL}: P={m[f'precision_{NEG_LABEL}']:.3f} "
               f"R={m[f'recall_{NEG_LABEL}']:.3f} "
               f"F1={m[f'f1_{NEG_LABEL}']:.3f} | "
               f"{POS_LABEL}: P={m[f'precision_{POS_LABEL}']:.3f} "
               f"R={m[f'recall_{POS_LABEL}']:.3f} "
               f"F1={m[f'f1_{POS_LABEL}']:.3f}")

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df = metrics_df[["classifier"]
        + [c for c in metrics_df.columns if c != "classifier"]]
    metrics_df.to_csv(out_dir / "final_test_metrics.csv", index=False)

    train_metrics_df = pd.DataFrame(train_metrics_rows)
    train_metrics_df = train_metrics_df[["classifier"]
        + [c for c in train_metrics_df.columns if c != "classifier"]]
    train_metrics_df.to_csv(out_dir / "final_train_metrics.csv", index=False)

    pd.DataFrame(gap_rows).to_csv(out_dir / "train_test_gap.csv", index=False)
    pd.DataFrame(hp_rows).to_csv(out_dir / "consensus_hp_used.csv", index=False)
    pd.DataFrame(gene_subset_rows).to_csv(
        out_dir / "gene_subsets_per_classifier.csv", index=False)

    # Per-class long-format CSV
    per_class_rows = []
    for _, r in metrics_df.iterrows():
        for cls in (NEG_LABEL, POS_LABEL):
            per_class_rows.append({
                "classifier": r["classifier"],
                "class":      cls,
                "precision":  float(r[f"precision_{cls}"]),
                "recall":     float(r[f"recall_{cls}"]),
                "f1":         float(r[f"f1_{cls}"]),
            })
    pd.DataFrame(per_class_rows).to_csv(
        out_dir / "per_class_metrics_test.csv", index=False)

    pred_rows = []
    for clf_name, p in predictions.items():
        for sid, yt, yp, yprob in zip(test_df.columns, p["y_true"],
                                       p["y_pred"], p["y_proba_pos"]):
            pred_rows.append({"classifier": clf_name, "sample_id": sid,
                               "y_true":  CLASS_NAMES[yt],
                               "y_pred":  CLASS_NAMES[yp],
                               f"proba_{POS_LABEL}": yprob,
                               f"proba_{NEG_LABEL}": 1.0 - yprob})
    pd.DataFrame(pred_rows).to_csv(
        out_dir / "final_test_predictions.csv", index=False)

    title_suffix = f"ES vs LS, top {panel_size}, SMOTE {variant}"
    plot_roc(predictions, out_dir / "roc_curves_test.png", title_suffix)
    plot_pr(predictions,  out_dir / "pr_curves_test.png",  title_suffix)
    plot_confusion_grid(predictions, out_dir / "confusion_matrices_test.png",
                         title_suffix)
    plot_metric_heatmap(metrics_df, out_dir / "metric_heatmap_test.png",
                         title_suffix)
    plot_per_class_metrics(metrics_df, out_dir / "per_class_metrics_test.png",
                           title_suffix)
    plot_train_vs_test(train_metrics_df, metrics_df,
                        out_dir / "train_vs_test_metrics.png", title_suffix)

    print(f"\n[{variant}] Outputs in: {out_dir}")
    return metrics_df, train_metrics_df


# ============================ Cross-variant comparison ======================
def run_comparison(metrics_by_variant, train_metrics_by_variant, panel_size):
    out_dir = OUTPUT_BASE / f"stage5_EVAL_ESLS_top{panel_size}_smote_comparison"
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 72)
    print(f"CROSS-VARIANT COMPARISON (top {panel_size})")
    print(f"Output dir: {out_dir.name}")
    print("=" * 72)

    rows = []
    for variant, df in metrics_by_variant.items():
        for _, r in df.iterrows():
            row = {"variant": variant, "panel_size": panel_size, **r.to_dict()}
            rows.append(row)
    combined = pd.DataFrame(rows)
    combined = combined[["variant", "panel_size", "classifier"]
        + [c for c in combined.columns
           if c not in ("variant", "panel_size", "classifier")]]
    combined.to_csv(out_dir / "test_metrics_all_variants.csv", index=False)

    if set(metrics_by_variant.keys()) == set(SMOTE_VARIANTS):
        a = metrics_by_variant["balanced"].set_index("classifier")
        b = metrics_by_variant["imbalanced"].set_index("classifier")
        common = sorted(set(a.index) & set(b.index))
        num_cols = [c for c in a.columns if c != "classifier"
                     and pd.api.types.is_numeric_dtype(a[c])]
        delta = (b.loc[common, num_cols] - a.loc[common, num_cols]).reset_index()
        delta.insert(0, "panel_size", panel_size)
        delta.to_csv(out_dir / "delta_imbalanced_minus_balanced.csv",
                      index=False)

    rows_t = []
    for variant, df in train_metrics_by_variant.items():
        for _, r in df.iterrows():
            rows_t.append({"variant": variant, "panel_size": panel_size,
                           **r.to_dict()})
    pd.DataFrame(rows_t).to_csv(
        out_dir / "train_metrics_all_variants.csv", index=False)

    plot_variant_comparison(
        metrics_by_variant,
        out_dir / "variant_comparison_test.png",
        title_suffix=f"top {panel_size}")

    rank_rows = []
    for variant, df in metrics_by_variant.items():
        sorted_df = df.sort_values("balanced_accuracy", ascending=False)
        for rank, (_, r) in enumerate(sorted_df.iterrows(), start=1):
            rank_rows.append({"panel_size": panel_size, "variant": variant,
                               "rank": rank,
                               "classifier": r["classifier"],
                               "balanced_accuracy": float(r["balanced_accuracy"]),
                               "macro_precision":   float(r["macro_precision"]),
                               "auroc":   float(r["auroc"]),
                               "macro_f1": float(r["macro_f1"])})
    pd.DataFrame(rank_rows).to_csv(
        out_dir / "classifier_ranking_per_variant.csv", index=False)

    print(f"\nComparison outputs in: {out_dir}")
    return combined, pd.DataFrame(rows_t)


def run_master_comparison(all_test_long: pd.DataFrame,
                           all_train_long: pd.DataFrame):
    out_dir = OUTPUT_BASE / "stage5_EVAL_ESLS_smote_master_comparison"
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 72)
    print("MASTER COMPARISON — all panel sizes x all variants")
    print(f"Output dir: {out_dir.name}")
    print("=" * 72)

    all_test_long.to_csv(
        out_dir / "test_metrics_all_panels_all_variants.csv", index=False)
    all_train_long.to_csv(
        out_dir / "train_metrics_all_panels_all_variants.csv", index=False)

    # Best (panel, variant, classifier) per headline metric — now includes
    # macro_precision in the headline list.
    headline = ["balanced_accuracy", "macro_precision", "macro_f1",
                "auroc", "pr_auc", "mcc"]
    best_rows = []
    for m in headline:
        if m not in all_test_long.columns:
            continue
        idx = all_test_long[m].astype(float).idxmax()
        r = all_test_long.loc[idx]
        best_rows.append({
            "metric":            m,
            "best_panel_size":   int(r["panel_size"]),
            "best_variant":      r["variant"],
            "best_classifier":   r["classifier"],
            "best_score":        float(r[m]),
        })
    pd.DataFrame(best_rows).to_csv(
        out_dir / "best_combination_per_metric.csv", index=False)

    if "balanced_accuracy" in all_test_long.columns:
        ba_wide = (all_test_long
                   .pivot_table(index=["variant", "classifier"],
                                 columns="panel_size",
                                 values="balanced_accuracy",
                                 aggfunc="first")
                   .reset_index())
        ba_wide.to_csv(
            out_dir / "balanced_accuracy_by_panel_size.csv", index=False)

        fig, ax = plt.subplots(figsize=(10, 6))
        for (variant, clf), sub in all_test_long.groupby(
                ["variant", "classifier"]):
            sub = sub.sort_values("panel_size")
            linestyle = "-" if variant == "balanced" else "--"
            ax.plot(sub["panel_size"].values,
                     sub["balanced_accuracy"].astype(float).values,
                     linestyle=linestyle,
                     marker="o",
                     color=CLF_COLORS.get(clf, "#444"),
                     label=f"{clf} ({variant})",
                     linewidth=1.6, alpha=0.9)
        ax.set_xticks(sorted(all_test_long["panel_size"].unique()))
        ax.set_xlabel("Panel size (top-K consensus genes)")
        ax.set_ylabel("Balanced accuracy (test)")
        ax.set_title("Balanced accuracy vs panel size — by classifier x SMOTE variant",
                      fontweight="bold")
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0),
                   fontsize=8, ncol=1)
        ax.grid(alpha=0.25, linestyle=":")
        ax.set_ylim([0, 1.05])
        fig.tight_layout()
        fig.savefig(out_dir / "balanced_accuracy_vs_panel_size.png",
                     bbox_inches="tight")
        plt.close(fig)

    # Per-(variant, classifier) macro-precision across panel sizes (new)
    if "macro_precision" in all_test_long.columns:
        prec_wide = (all_test_long
                     .pivot_table(index=["variant", "classifier"],
                                   columns="panel_size",
                                   values="macro_precision",
                                   aggfunc="first")
                     .reset_index())
        prec_wide.to_csv(
            out_dir / "macro_precision_by_panel_size.csv", index=False)

    print(f"\nMaster comparison outputs in: {out_dir}")


def main():
    print("=" * 72)
    print("STAGE 5 (ES vs LS) SMOTE — Multi-panel test eval")
    print(f"Panel sizes  : {PANEL_SIZES}")
    print(f"Variants     : {SMOTE_VARIANTS}")
    print(f"Total runs   : {len(PANEL_SIZES)} panels x {len(SMOTE_VARIANTS)} "
          f"variants = {len(PANEL_SIZES) * len(SMOTE_VARIANTS)}")
    print("=" * 72)

    all_test_long_frames  = []
    all_train_long_frames = []

    for panel_size in PANEL_SIZES:
        metrics_by_variant       = {}
        train_metrics_by_variant = {}
        for variant in SMOTE_VARIANTS:
            m_test, m_train = run_variant(variant, panel_size)
            metrics_by_variant[variant]       = m_test
            train_metrics_by_variant[variant] = m_train

        test_long, train_long = run_comparison(
            metrics_by_variant, train_metrics_by_variant, panel_size)
        all_test_long_frames.append(test_long)
        all_train_long_frames.append(train_long)

    run_master_comparison(
        pd.concat(all_test_long_frames,  ignore_index=True),
        pd.concat(all_train_long_frames, ignore_index=True))


if __name__ == "__main__":
    main()


STAGE 5 (ES vs LS) SMOTE — Multi-panel test eval
Panel sizes  : [5, 7, 10, 12, 15]
Variants     : ['balanced', 'imbalanced']
Total runs   : 5 panels x 2 variants = 10

VARIANT: SMOTE = BALANCED  (top 5)
Stage-4 source: stage4_HP_ESLS_top5_smote_balanced
Output dir    : stage5_EVAL_ESLS_top5_smote_balanced
Train (SMOTE-augmented): 82 samples x 5 genes (11 synthetic)
Test (held-out)        : 31 samples x 5 genes
Train class balance: LS=41, ES=41
Test  class balance: LS=12, ES=19
Sample weights: real=71 (composite sex x cell_type x class), synthetic=11 (uniform 1.0)  -> mean=1.000, min=0.509, max=4.640

Tailored gene subsets at fold_inclusion_pct >= 80.0%:
  ElasticNet: 4 genes
  mSVM-RFE: 3 genes

[mSVM-RBF] full panel (5 genes) | applied: {'clf__C': 10.0, 'clf__gamma': 0.1}
  TRAIN(+SMOTE) BA=0.866 AUROC=0.938  macro-Prec=0.866 F1=0.866
  TEST          BA=0.645 AUROC=0.768  macro-Prec=0.657 F1=0.648 MCC=0.302
  TEST per-class — LS: P=0.600 R=0.500 F1=0.545 | ES: P=0.714 R=0.789 F1=0.750